# 03b — Two-Stage (lgbm) 임시 단일 노트북 — **rev.002 (mean agg)**

**목적**: 표준 Two-Stage(분류+회귀) 한 번 돌려서 BagZIT plateau(val 0.00571)에 들어오는지 확인. **일회용**.

**rev.002 변경점 (vs rev.001)**:
- rev.001: die→unit `sum` 집계 + 30-931-001 (yeo-johnson) PP/HP. broadcast 학습이라 sum이 ~4× 오버슈트 → val 0.0098 (plateau 밖)
- **rev.002**: die→unit `mean` 집계 + `reg-lgbm-001` (log1p) best PP/HP 그대로 차용. Stage 1 분류만 추가. 비교 기준선 reg_only/lgbm는 val 0.005731 / test 0.008429

**구성**:
- 전처리 = `4_output/final/reg_only/lgbm/best_params.json` `effective_pp_params` (log1p 프리셋) 그대로
- Stage 1 (분류): LGBMClassifier(objective='binary'), HP는 reg-lgbm-001 base 차용
- Stage 2 (회귀): LGBMRegressor(objective='poisson'), HP = reg-lgbm-001 best, **target = log1p(y), y>0 die만**
- 최종 die pred = `P(y>0) × expm1(reg_log)`, **unit pred = mean(die pred)** (reg_only/lgbm와 동일 집계)

**격리**: `4_output/_temp/two_stage_temp/` 신규. 모듈 무수정.

**비교 대상**:
- reg_only/lgbm 단독: val 0.005731 / test 0.008429 (Stage 1 없음)
- BagZIT plateau best: val 0.005701 (stacking_11base), 단일 0.005708 (zit_only)
- 본 노트북이 reg_only/lgbm보다 좋아지면 Stage 1 분류 효과 확인 가능

## 1. 환경 + import

In [1]:
import os, sys, json

%run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import (
    PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR,
)
from utils.data import load_all, get_feat_cols, split_xs

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess

import lightgbm as lgb
from sklearn.model_selection import KFold

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트


## 2. 설정 (reg-lgbm-001 best PP/HP, log1p, mean 집계)

`4_output/final/reg_only/lgbm/best_params.json`에서 PP(`effective_pp_params`)와 HP(`best_params_resolved`) 그대로 차용.
- HPO best OOF = 0.005521 (reg_only 단독), val 0.005731 / test 0.008429
- objective: poisson (Stage 2 회귀에 그대로 사용; Stage 1 분류는 binary로 override)

In [2]:
EXP_ID  = 'two-stage-temp-002'
N_FOLDS = 5
CLIP_Y_EXTREME = True
TARGET_TRANSFORM = 'log1p'   # ★ reg-lgbm-001 동일

OUT_DIR = os.path.join(OUTPUT_DIR, '_temp', 'two_stage_temp')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 전처리 PARAMS (reg-lgbm-001 effective_pp_params, log1p 프리셋) ──
# 02_reg_only.ipynb의 PARAMS_BY_TRANSFORM['log1p']와 동일
PARAMS = {
    'missing_threshold':          0.5,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',           # ★ rev.001의 'target_corr' (leakage 옵트인) 폐기
    'add_indicator':              True,
    'indicator_threshold':        0.25,
    'spatial_max_dist':           5.0,
    'post_impute_corr_threshold': 0.99,
    'post_impute_corr_keep_by':   'std',
}

# ── LGBM HP (reg-lgbm-001 best_params_resolved) ──
# objective는 두 stage에서 다르게 쓰므로 LGB_HP_BASE에서 분리
LGB_HP_BASE = dict(
    n_estimators       = 957,
    learning_rate      = 0.006021010944246521,
    num_leaves         = 379,
    max_depth          = 10,
    min_child_samples  = 343,
    subsample          = 0.7112499255653136,
    subsample_freq     = 1,
    colsample_bytree   = 0.5969486967925332,
    reg_alpha          = 0.008404729186766218,
    reg_lambda         = 0.0001507026474554505,
    min_split_gain     = 0.00010162647954934355,
    path_smooth        = 24.81165187056286,
    random_state       = SEED,
    n_jobs             = -1,
    verbose            = -1,
)
REG_OBJECTIVE = 'poisson'   # ★ reg-lgbm-001 best objective (Stage 2)
CLF_OBJECTIVE = 'binary'    # Stage 1 분류 고정

print(f'EXP_ID={EXP_ID} | N_FOLDS={N_FOLDS}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM}')
print(f'OUT_DIR={OUT_DIR}')
print(f'PARAMS keys: {list(PARAMS)}')
print(f'LGB_HP_BASE keys: {len(LGB_HP_BASE)} | reg objective={REG_OBJECTIVE}, clf objective={CLF_OBJECTIVE}')

EXP_ID=two-stage-temp-002 | N_FOLDS=5
TARGET_TRANSFORM=log1p
OUT_DIR=c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\two_stage_temp
PARAMS keys: ['missing_threshold', 'corr_threshold', 'corr_keep_by', 'add_indicator', 'indicator_threshold', 'spatial_max_dist', 'post_impute_corr_threshold', 'post_impute_corr_keep_by']
LGB_HP_BASE keys: 15 | reg objective=poisson, clf objective=binary


## 3. 데이터 로드 + Y clip

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개 샘플')

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

print(f'\n[데이터 로드] xs={xs.shape}, X feat_cols={len(feat_cols)}')
print(f'  unit train={len(y_train_unit):,}, val={len(y_val_unit):,}, test={len(y_test_unit):,}')
print(f'  y_train: max={y_train_unit.max():.6f}, mean={y_train_unit.mean():.6f}, zero ratio={(y_train_unit==0).mean():.1%}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개 샘플

[데이터 로드] xs=(174572, 1091), X feat_cols=1087
  unit train=26,187, val=8,727, test=8,729
  y_train: max=0.097417, mean=0.002481, zero ratio=70.8%


## 4. 전처리 (reg-lgbm-001 log1p 프리셋) + numpy 변환

rev.001은 30-931-001 (yeo-johnson)을 썼지만 rev.002는 log1p 프리셋이라 cleaning 결과가 더 공격적 (reg_only/lgbm 기준 ~568 features 예상, rev.001은 761).

In [4]:
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

print(f'\n[cleaning] feat_cols_clean={len(feat_cols_clean)}')
print(f'  xs_train_die: {xs_train_die.shape}, val: {xs_val_die.shape}, test: {xs_test_die.shape}')

X_train_die = xs_train_die[feat_cols_clean].values.astype(np.float64)
X_val_die   = xs_val_die[feat_cols_clean].values.astype(np.float64)
X_test_die  = xs_test_die[feat_cols_clean].values.astype(np.float64)
uid_train_die = xs_train_die[KEY_COL].values
uid_val_die   = xs_val_die[KEY_COL].values
uid_test_die  = xs_test_die[KEY_COL].values
y_train_die_broadcast = pd.Series(uid_train_die).map(y_train_unit).values.astype(np.float64)
assert not pd.isna(y_train_die_broadcast).any(), 'unmapped train die y'

# binary target broadcast (Stage 1 분류용)
y_bin_die_broadcast = (y_train_die_broadcast > 0).astype(np.int32)
n_train_die = len(X_train_die)
n_val_die   = len(X_val_die)
n_test_die  = len(X_test_die)
print(f'\n  X_train_die: {X_train_die.shape}, val: {X_val_die.shape}, test: {X_test_die.shape}')
print(f'  y_train_die (broadcasted unit y): mean={y_train_die_broadcast.mean():.6f}')
print(f'  y_bin_die (broadcasted y>0):      pos ratio={y_bin_die_broadcast.mean():.4f}')

[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=50%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 332개, 잔여: 564개
    컬럼: 896 → 564 (332개 제거)
    DataFrame: (104748, 622)

[결측 indicator] 4개 컬럼 추가 (결측률 >= 25%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, dist<=5.0): 156,772개 채움 → 잔여: 186,722
  2단계 (lot 평균, train 기준): 105,526개 채움 → 잔여: 81,196
  3단계 (train 전체 평균): 81,196개 채움 → 잔여: 0

  [요약] 343,494 → 공간(156,772) → lot(105,526) → 전체(81,196) → 잔여(0)

[고상관 제거] threshold=0.99, keep_by=std (std)
  제거: 0개, 잔여: 564개
    [고상관 제거 2차 / imputation 후] threshold=0.99
    컬럼: 564 → 564 (0개 제거)
    DataFrame: (104748, 626)

클리닝 완

## 5. KFold split (BagZIT 노트북과 동일 — shuffle=True, random_state=SEED)

stacking pool 호환 위해 동일 fold split.

In [5]:
unit_ids_train_unique = y_train_unit.index.values
kf_global = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf_global.split(unit_ids_train_unique))
print(f'fold split: {N_FOLDS} folds')
for fi, (tr_idx, vl_idx) in enumerate(FOLDS):
    print(f'  fold {fi+1}: train={len(tr_idx):,} val={len(vl_idx):,}')

fold split: 5 folds
  fold 1: train=20,949 val=5,238
  fold 2: train=20,949 val=5,238
  fold 3: train=20,950 val=5,237
  fold 4: train=20,950 val=5,237
  fold 5: train=20,950 val=5,237


## 6. 5-fold Two-Stage 학습 + die-level 캡쳐

각 fold:
1. **Stage 1 분류**: LGBMClassifier(objective='binary'), target = (y_die > 0) (broadcast된 binary), 모든 train die
2. **Stage 2 회귀**: LGBMRegressor(objective='poisson'), target = log1p(y_die), **y_die > 0 die만**
3. **최종 die pred** = `prob_die × expm1(reg_die)`
4. **unit pred** = same-unit die-level final **mean** 집계 (reg_only/lgbm와 동일 — broadcast 학습이라 sum은 4× 오버슈트, mean이 정합)

In [6]:
import time

def _mean_die_to_unit(pred_die, uid_die):
    """die-level → unit-level mean 집계. reg_only/lgbm와 동일."""
    unit_id = np.asarray(uid_die)
    unique_units, inverse = np.unique(unit_id, return_inverse=True)
    n_units = len(unique_units)
    pred_sum = np.zeros(n_units)
    cnt      = np.zeros(n_units)
    np.add.at(pred_sum, inverse, pred_die)
    np.add.at(cnt,      inverse, 1.0)
    pred_unit = pred_sum / cnt
    return pred_unit, unique_units

# die-level 캡쳐
oof_die_prob   = np.full(n_train_die, np.nan)
oof_die_reg    = np.full(n_train_die, np.nan)
oof_die_pred   = np.full(n_train_die, np.nan)

val_die_prob   = np.zeros(n_val_die)
val_die_reg    = np.zeros(n_val_die)
val_die_pred   = np.zeros(n_val_die)

test_die_prob  = np.zeros(n_test_die)
test_die_reg   = np.zeros(n_test_die)
test_die_pred  = np.zeros(n_test_die)

print('=== 5-fold Two-Stage 학습 ===')
t0 = time.time()
for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
    tr_units = unit_ids_train_unique[tr_uidx]
    vl_units = unit_ids_train_unique[vl_uidx]
    tr_die_mask = np.isin(uid_train_die, tr_units)
    vl_die_mask = np.isin(uid_train_die, vl_units)

    X_tr  = X_train_die[tr_die_mask]
    X_vl  = X_train_die[vl_die_mask]
    y_tr  = y_train_die_broadcast[tr_die_mask]
    yb_tr = y_bin_die_broadcast[tr_die_mask]

    # ── Stage 1 분류 (LGBMClassifier, binary) ──
    clf = lgb.LGBMClassifier(**LGB_HP_BASE, objective=CLF_OBJECTIVE)
    clf.fit(X_tr, yb_tr)

    # ── Stage 2 회귀 (y>0 die만, log1p target, poisson objective) ──
    pos_mask = y_tr > 0
    if pos_mask.sum() < 100:
        raise RuntimeError(f'fold {fold_idx+1}: y>0 die 수가 너무 적음 ({pos_mask.sum()})')
    y_tr_pos_log = np.log1p(y_tr[pos_mask])
    reg = lgb.LGBMRegressor(**LGB_HP_BASE, objective=REG_OBJECTIVE)
    reg.fit(X_tr[pos_mask], y_tr_pos_log)

    # ── 예측 (vl, val, test) ──
    def _predict(Xs):
        prob = clf.predict_proba(Xs)[:, 1]
        prob = np.clip(prob, 0.0, 1.0)
        reg_log = reg.predict(Xs)
        reg_y   = np.clip(np.expm1(reg_log), 0.0, None)   # log1p 역변환 + 음수 clip
        final   = prob * reg_y
        return prob, reg_y, final

    p_vl, r_vl, f_vl = _predict(X_vl)
    p_v,  r_v,  f_v  = _predict(X_val_die)
    p_t,  r_t,  f_t  = _predict(X_test_die)

    # OOF (vl_units)
    oof_die_prob[vl_die_mask] = p_vl
    oof_die_reg[vl_die_mask]  = r_vl
    oof_die_pred[vl_die_mask] = f_vl

    # val/test 5-fold avg
    val_die_prob  += p_v / N_FOLDS
    val_die_reg   += r_v / N_FOLDS
    val_die_pred  += f_v / N_FOLDS
    test_die_prob += p_t / N_FOLDS
    test_die_reg  += r_t / N_FOLDS
    test_die_pred += f_t / N_FOLDS

    print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s) — '
          f'fold pos die ratio={pos_mask.mean():.3f}, prob_vl mean={p_vl.mean():.4f}, '
          f'reg_vl mean={r_vl.mean():.6f}')

assert not np.isnan(oof_die_prob).any(), 'OOF die prob 미커버'
assert not np.isnan(oof_die_reg).any(),  'OOF die reg 미커버'
assert not np.isnan(oof_die_pred).any(), 'OOF die pred 미커버'

print(f'\n[학습 완료] {time.time()-t0:.0f}s')

=== 5-fold Two-Stage 학습 ===
  fold 1/5 done (67s) — fold pos die ratio=0.290, prob_vl mean=0.2897, reg_vl mean=0.008437
  fold 2/5 done (130s) — fold pos die ratio=0.294, prob_vl mean=0.2898, reg_vl mean=0.008481
  fold 3/5 done (196s) — fold pos die ratio=0.293, prob_vl mean=0.2915, reg_vl mean=0.008481
  fold 4/5 done (262s) — fold pos die ratio=0.292, prob_vl mean=0.2933, reg_vl mean=0.008516
  fold 5/5 done (326s) — fold pos die ratio=0.291, prob_vl mean=0.2907, reg_vl mean=0.008458

[학습 완료] 326s


## 7. unit aggregate + RMSE

In [7]:
oof_unit_arr,  oof_unit_ids  = _mean_die_to_unit(oof_die_pred,  uid_train_die)
val_unit_arr,  val_unit_ids  = _mean_die_to_unit(val_die_pred,  uid_val_die)
test_unit_arr, test_unit_ids = _mean_die_to_unit(test_die_pred, uid_test_die)

oof_unit  = pd.Series(oof_unit_arr,  index=oof_unit_ids).reindex(y_train_unit.index)
val_unit  = pd.Series(val_unit_arr,  index=val_unit_ids).reindex(y_val_unit.index)
test_unit = pd.Series(test_unit_arr, index=test_unit_ids).reindex(y_test_unit.index)

def _rmse(pred, true):
    return float(np.sqrt(np.mean((pred.values - true.values) ** 2)))

oof_rmse  = _rmse(oof_unit,  y_train_unit)
val_rmse  = _rmse(val_unit,  y_val_unit)
test_rmse = _rmse(test_unit, y_test_unit)

print('=' * 75)
print(f'  Two-Stage temp (lgbm only, reg-lgbm-001 PP/HP, log1p, mean agg)')
print('=' * 75)
print(f'  {"":12s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"unit RMSE":12s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
print(f'  비교 기준선:')
print(f'    reg_only/lgbm (Stage 1 없음):     val=0.005731, test=0.008429')
print(f'    BagZIT plateau best (zit_only):   val=0.005709, test=0.008414')
print(f'    Stacking 11-base (val best):       val=0.005701, test=0.008408')
print('=' * 75)
if val_rmse < 0.0058:
    print('  → plateau 영역 안. stacking pool에 추가 가능.')
elif val_rmse < 0.006:
    print('  → plateau 근처. residual 패턴이 다르면 stacking에 도움 가능.')
else:
    print('  → plateau 밖. 본격 HPO 또는 구조 수정 필요.')

  Two-Stage temp (lgbm only, reg-lgbm-001 PP/HP, log1p, mean agg)
                        OOF          val         test
  unit RMSE        0.005511     0.005718     0.008417
---------------------------------------------------------------------------
  비교 기준선:
    reg_only/lgbm (Stage 1 없음):     val=0.005731, test=0.008429
    BagZIT plateau best (zit_only):   val=0.005709, test=0.008414
    Stacking 11-base (val best):       val=0.005701, test=0.008408
  → plateau 영역 안. stacking pool에 추가 가능.


## 8. 산출물 저장 (`_temp/two_stage_temp/`)

In [8]:
def _build_die_df(uid_arr, die_id_arr, position_arr, prob, reg, pred, y_unit):
    df = pd.DataFrame({
        KEY_COL:     uid_arr,
        DIE_KEY_COL: die_id_arr,
        'position':  position_arr,
        'prob':      prob,
        'reg':       reg,
        'pred':      pred,
    })
    if y_unit is not None:
        df[TARGET_COL] = df[KEY_COL].map(y_unit)
    return df

_build_die_df(
    uid_train_die, xs_train_die[DIE_KEY_COL].values, xs_train_die['position'].values,
    oof_die_prob, oof_die_reg, oof_die_pred, y_train_unit,
).to_csv(os.path.join(OUT_DIR, 'oof_die.csv'), index=False)
_build_die_df(
    uid_val_die, xs_val_die[DIE_KEY_COL].values, xs_val_die['position'].values,
    val_die_prob, val_die_reg, val_die_pred, y_val_unit,
).to_csv(os.path.join(OUT_DIR, 'val_die.csv'), index=False)
_build_die_df(
    uid_test_die, xs_test_die[DIE_KEY_COL].values, xs_test_die['position'].values,
    test_die_prob, test_die_reg, test_die_pred, y_test_unit,
).to_csv(os.path.join(OUT_DIR, 'test_die.csv'), index=False)

def _build_unit_df(unit_pred, y_unit):
    return pd.DataFrame({
        KEY_COL: unit_pred.index.values,
        'pred':  unit_pred.values,
        'health': y_unit.reindex(unit_pred.index).values,
    })

_build_unit_df(oof_unit,  y_train_unit).to_csv(os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(val_unit,  y_val_unit ).to_csv(os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(test_unit, y_test_unit).to_csv(os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

meta = {
    'exp_id':            EXP_ID,
    'model':             'Two-Stage (lgbm clf binary + lgbm reg poisson) + log1p + mean agg',
    'target_transform':  TARGET_TRANSFORM,
    'aggregation':       'mean (die→unit)',
    'n_folds':           N_FOLDS,
    'oof_rmse':          oof_rmse,
    'val_rmse':          val_rmse,
    'test_rmse':         test_rmse,
    'preprocess_PARAMS': PARAMS,
    'effective_pp_params': pp['effective_params'],
    'lgb_hp_base':       {k: v for k, v in LGB_HP_BASE.items() if k not in ['random_state', 'n_jobs', 'verbose']},
    'reg_objective':     REG_OBJECTIVE,
    'clf_objective':     CLF_OBJECTIVE,
    'pp_source':         'reg-lgbm-001 best_params.json effective_pp_params (log1p 프리셋)',
    'hp_source':         'reg-lgbm-001 best_params.json best_params_resolved',
    'CLIP_Y_EXTREME':    CLIP_Y_EXTREME,
    'feat_cols_clean_n': len(feat_cols_clean),
    'SEED':              int(SEED),
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:25s}  {sz:>10,.1f} KB')

저장 완료: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\two_stage_temp
  meta.json                         1.9 KB
  oof_die.csv                   9,915.9 KB
  oof_unit.csv                    970.4 KB
  test_die.csv                  3,304.5 KB
  test_unit.csv                   323.5 KB
  val_die.csv                   3,304.5 KB
  val_unit.csv                    323.5 KB


## 9. 요약

In [9]:
print('=' * 75)
print(f' Two-Stage temp (lgbm) — rev.002 결과 요약')
print('=' * 75)
print(f'  EXP_ID            : {EXP_ID}')
print(f'  PP source         : reg-lgbm-001 effective_pp_params (log1p 프리셋)')
print(f'  HP source         : reg-lgbm-001 best_params_resolved (objective: poisson)')
print(f'  target transform  : {TARGET_TRANSFORM}')
print(f'  die→unit agg      : mean')
print(f'  feat cols (clean) : {len(feat_cols_clean)}')
print('-' * 75)
print(f'  {"":10s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"unit RMSE":10s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
print(f'  → reg_only/lgbm 단독 (val=0.005731) 대비 개선되면 Stage 1 분류 효과.')
print(f'  → val/test 가 plateau(0.0057x/0.0084x) 안에 들어오면 stacking 후보.')
print(f'  → residual corr 가 BagZIT 변형들과 다르면 stacking 효과 기대.')
print('=' * 75)

 Two-Stage temp (lgbm) — rev.002 결과 요약
  EXP_ID            : two-stage-temp-002
  PP source         : reg-lgbm-001 effective_pp_params (log1p 프리셋)
  HP source         : reg-lgbm-001 best_params_resolved (objective: poisson)
  target transform  : log1p
  die→unit agg      : mean
  feat cols (clean) : 568
---------------------------------------------------------------------------
                      OOF          val         test
  unit RMSE      0.005511     0.005718     0.008417
---------------------------------------------------------------------------
  → reg_only/lgbm 단독 (val=0.005731) 대비 개선되면 Stage 1 분류 효과.
  → val/test 가 plateau(0.0057x/0.0084x) 안에 들어오면 stacking 후보.
  → residual corr 가 BagZIT 변형들과 다르면 stacking 효과 기대.
